<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/09-serving-inference/03-gpu-capacity-math.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU Capacity Math (concept + measurement)

**Goal:** Given a model and an accelerator (an H100, an H20, a domestic part such as Alibaba's PPU or Huawei's Ascend), answer the three questions every self-hosting plan starts with: **does it fit, how many tokens per second, and how many cards for the target load?** Then measure a real GPU and see how close the estimate gets.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Why this notebook exists

[Notebook 02](02-inference-performance.ipynb) sized a fleet from a *measured* tokens-per-second number. This notebook derives that number from the spec sheet, so you can compare accelerators **before** you have one, and so you can tell which spec matters for which workload.

Everything above the appendix is arithmetic and runs on a CPU. The optional appendix measures bandwidth and FLOP/s on a Colab T4, then checks the prediction against a real vLLM server.

> **⭐ Key takeaway —** capacity and speed come from **different specs**. Memory *capacity* decides what fits and how many requests run at once; memory *bandwidth* decides decode speed; tensor *FLOP/s* decide prefill speed (and decode too, on compute-poor cards). Work in order: fit → batch ceiling → decode → prefill → latency check → fleet.

## The inputs: spec sheet and model config

Specs as of 2026-11, **dense** tensor FLOP/s (many listings print the 2× "sparsity" number; never mix the two). `None` means not published or not supported. The domestic parts are left out of the table on purpose: their bandwidth and FLOP/s are estimates or unpublished, and we treat them as parameters below.

The model configs are the four numbers the cache needs (`num_hidden_layers`, `num_key_value_heads`, `head_dim`, and the parameter count), read from each model's `config.json`.

In [ ]:
import math

# name: (HBM GB, bandwidth TB/s, dense BF16 TFLOP/s, dense FP8 TFLOP/s)   as of 2026-11
GPUS = {
    "T4":        (16,  0.32,  65,   None),   # free Colab; FP16 figure, no FP8
    "L4":        (24,  0.30,  121,  242),
    "L40S":      (48,  0.864, 362,  733),
    "A100-80GB": (80,  2.04,  312,  None),
    "H100-SXM":  (80,  3.35,  989,  1979),
    "H20":       (96,  4.0,   148,  296),    # third-party listings; a 141 GB variant also exists
    "H200":      (141, 4.8,   989,  1979),
    "B200":      (180, 8.0,   2250, 4500),
}

# Model shapes (as of 2026-11): params in billions, layers, KV heads, head dim; MoE adds active params.
MODELS = {
    "8B":   dict(params=8.03,  layers=32, kv_heads=8, head_dim=128),                 # Llama-3.1-8B shape
    "32B":  dict(params=32.8,  layers=64, kv_heads=8, head_dim=128),                 # Qwen3-32B shape
    "70B":  dict(params=70.6,  layers=80, kv_heads=8, head_dim=128),                 # Llama-3.1-70B shape
    "235B": dict(params=235.1, layers=94, kv_heads=4, head_dim=128, active=22.0),    # Qwen3-235B-A22B shape (MoE)
}

BYTES = {"BF16": 2.0, "FP8": 1.0, "INT4": 0.5625}   # INT4 includes group scales

# Efficiency assumptions: replace with measured values (see the appendix).
USABLE = 0.90        # vLLM's default --gpu-memory-utilization
OVERHEAD_GB = 2.0    # activations, CUDA graphs, runtime, per GPU; read the real number from the server log
MBU = 0.80           # achievable share of memory bandwidth
MFU = 0.50           # achievable share of tensor FLOP/s

for name, (hbm, bw, bf16, fp8) in GPUS.items():
    print(f"{name:<10} {hbm:>4} GB  {bw:>5.2f} TB/s  BF16 {bf16:>5} TF  FP8 {str(fp8):>5} TF  ridge {bf16 / bw:>4.0f} FLOP/byte")

The last column is the **ridge point**: FLOP/s ÷ bandwidth. A BF16 decode step at batch *B* does about *B* FLOPs per byte it reads, so below the ridge the card waits on memory and above it the card waits on math. Most data-center GPUs have ridges of 150–300; **the H20's is 37**. Hold that thought until step 4.

## Step 1 — does it fit?

```text
usable memory  = 0.9 × HBM
weights        = parameters × bytes per parameter
KV pool        = usable − weights − ~2 GB overhead          (per GPU; × TP size for a TP group)
KV per token   = 2 (K and V) × layers × KV heads × head dim × bytes per value
```

Note **KV heads**, not attention heads: grouped-query attention (GQA) shares one K/V head across several query heads, so modern models cache far less than their hidden size suggests.

In [ ]:
def weights_gb(model, prec):
    return MODELS[model]["params"] * BYTES[prec]

def kv_mb_per_token(model, kv_bytes=2.0):
    c = MODELS[model]
    return 2 * c["layers"] * c["kv_heads"] * c["head_dim"] * kv_bytes / 1e6

def kv_pool_gb(gpu, model, prec, tp=1):
    return tp * (USABLE * GPUS[gpu][0] - OVERHEAD_GB) - weights_gb(model, prec)

for m in MODELS:
    print(f"{m:>5}: {kv_mb_per_token(m):.3f} MB of BF16 KV cache per token")

COMBOS = [("8B", "BF16"), ("8B", "FP8"), ("32B", "BF16"), ("32B", "FP8"), ("70B", "FP8"), ("70B", "INT4"), ("235B", "FP8")]
SHOW = ["L4", "L40S", "H100-SXM", "H20", "H200", "B200"]

print(f"\n{'model':<10}{'weights':>9}" + "".join(f"{g:>17}" for g in SHOW))
for m, p in COMBOS:
    cells = []
    for g in SHOW:
        pool = kv_pool_gb(g, m, p)
        cells.append("no" if pool < 4 else f"{pool:.0f}GB/{pool * 1e3 / kv_mb_per_token(m) / 1e3:.0f}K tok")
    print(f"{m + ' ' + p:<10}{weights_gb(m, p):>7.1f}GB" + "".join(f"{c:>17}" for c in cells))

Each cell is the KV pool left on **one** card and how many tokens of BF16 cache it holds ("no" = under 4 GB, too little to serve).

- A 70B model in FP8 doesn't fit one H100 with any cache, fits one **H20 or PPU (both 96 GB)** with ~42K tokens, and fits one H200 comfortably.
- The 235B MoE fits nowhere on one card: **all experts must be resident** even though only 22B are active per token. It needs a tensor-parallel group.
- Fit depends only on capacity, so it's the one step you can finish for an accelerator whose other specs are unpublished.

> **🚩 Common mistake —** fitting the *weights* and stopping. The server starts, then serves two requests at a time, because there's no room left for the cache. Fit weights **plus** overhead **plus** the cache for the batch you need.

In [ ]:
# The MoE model on TP groups: how much cache is left?
for g, tp in [("H200", 2), ("H100-SXM", 4), ("B200", 2)]:
    print(f"235B FP8 on {tp} x {g}: KV pool {kv_pool_gb(g, '235B', 'FP8', tp):.0f} GB")

## Step 2 — how fast is decode?

Every decode step reads all weights once, plus the cache of every sequence in the batch, and does `2 × active params` FLOPs per sequence. The step takes whichever is longer:

```text
memory time   = (weights + batch × context × KV per token) / (bandwidth × MBU)
compute time  = 2 × active params × batch / (FLOP/s × MFU)
tokens/s      = batch / max(memory time, compute time)
```

In [ ]:
def tflops(gpu, prec):
    bf16, fp8 = GPUS[gpu][2], GPUS[gpu][3]
    return fp8 if prec in ("FP8", "INT4") and fp8 else bf16

def decode_step_ms(gpu, model, prec, batch, ctx, tp=1, kv_bytes=2.0):
    """Return (step ms, memory ms, compute ms) for one decode step."""
    bw = GPUS[gpu][1] * 1e12 * tp * MBU
    kv_gb = batch * ctx * kv_mb_per_token(model, kv_bytes) / 1e3
    mem_ms = (weights_gb(model, prec) + kv_gb) * 1e9 / bw * 1e3
    active = MODELS[model].get("active", MODELS[model]["params"])
    comp_ms = 2 * active * 1e9 * batch / (tflops(gpu, prec) * 1e12 * tp * MFU) * 1e3
    return max(mem_ms, comp_ms), mem_ms, comp_ms

print("Batch-1 decode, 8B model:")
for g in ["L4", "H100-SXM", "H20", "H200"]:
    bf = 1000 / decode_step_ms(g, "8B", "BF16", 1, 0)[0]
    f8 = 1000 / decode_step_ms(g, "8B", "FP8", 1, 0)[0]
    print(f"  {g:<9} BF16 {bf:>4.0f} tok/s   FP8 {f8:>4.0f} tok/s")

print("\nBatching the 8B BF16 model on an H100, 2,000 tokens of context each:")
print(f"  {'batch':>5} {'step ms':>8} {'mem ms':>7} {'comp ms':>8} {'tok/s':>7}")
for b in [1, 8, 32, 64, 128, 200]:
    s, mem, comp = decode_step_ms("H100-SXM", "8B", "BF16", b, 2000)
    print(f"  {b:>5} {s:>8.1f} {mem:>7.1f} {comp:>8.1f} {b / s * 1000:>7.0f}")

ceiling = GPUS["H100-SXM"][1] * 1e12 * MBU / (2000 * kv_mb_per_token("8B") * 1e6)
print(f"\n  cache-read ceiling: bandwidth / (context x KV per token) = {ceiling:,.0f} tok/s")

Two lessons:

1. **At batch 1 only bandwidth matters.** The H20 beats the H100 on a single user, because it has more bandwidth. Its weak FLOP/s don't show yet.
2. **Batching multiplies throughput, then flattens.** One weight read serves the whole batch, so tokens/s climbs ~48× while the step only quadruples. But the cache reads grow with the batch, and tokens/s approaches `bandwidth / (context × KV per token)`, about 10,200 here. That ceiling is why GQA, an FP8 KV cache, and shorter contexts raise throughput: fewer bytes per sequence per step.

## Step 3 — how fast is prefill?

Prefill multiplies the whole prompt through every weight in one pass: ~`2 × active params × prompt tokens` FLOPs, compute-bound on every card. It's the floor on time to first token (TTFT), before any queueing.

In [ ]:
def prefill_s(gpu, model, prec, prompt_tokens, tp=1):
    active = MODELS[model].get("active", MODELS[model]["params"])
    return 2 * active * 1e9 * prompt_tokens / (tflops(gpu, prec) * 1e12 * tp * MFU)

print("Prefill of a 2,000-token prompt, 8B BF16 (the TTFT floor):")
for g in ["H100-SXM", "H200", "H20", "L4"]:
    print(f"  {g:<9} {prefill_s(g, '8B', 'BF16', 2000) * 1000:>5.0f} ms")

The H20 that *won* batch-1 decode is now 6.7× slower than an H100. Which spec bounds a workload depends on its **ratio of prompt tokens to output tokens**.

> **💡 Why —** prefill pushes thousands of prompt tokens through each weight it reads (thousands of FLOPs per byte, so the tensor cores saturate); decode pushes one token per sequence per weight read (*batch* FLOPs per byte, so it waits on memory).

## Step 4 — replica capacity and the latency check

A replica's capacity comes from both phases: prefill can serve `1 / prefill time` requests/s, decode can serve `decode tokens/s / output tokens`. If the server overlaps them perfectly (chunked prefill), capacity is the **smaller** of the two; if it never overlaps them, it's `1 / (sum of per-request times)`. Real servers sit between, so we report both.

Then the latency check: the decode step **is** the TPOT, so if it exceeds the target we shrink the batch (vLLM's `--max-num-seqs`) until it fits.

In [ ]:
def replica_capacity(gpu, model, prec, prompt, output, tp=1, kv_bytes=2.0, tpot_slo_ms=None, max_seqs=256):
    pool = kv_pool_gb(gpu, model, prec, tp)
    per_seq_gb = (prompt + output) * kv_mb_per_token(model, kv_bytes) / 1e3   # reserve the full length
    avg_ctx = prompt + output / 2                                             # average context while decoding
    batch = min(max_seqs, int(pool / per_seq_gb))
    while tpot_slo_ms and batch > 1 and decode_step_ms(gpu, model, prec, batch, avg_ctx, tp, kv_bytes)[0] > tpot_slo_ms:
        batch -= 1
    step, mem, comp = decode_step_ms(gpu, model, prec, batch, avg_ctx, tp, kv_bytes)
    decode_tok_s = batch / (step / 1000)
    ttft = prefill_s(gpu, model, prec, prompt, tp)
    pre_cap, dec_cap = 1 / ttft, decode_tok_s / output
    return dict(pool_gb=pool, batch=batch, step_ms=step, bound="memory" if mem >= comp else "compute",
                decode_tok_s=decode_tok_s, ttft_floor_s=ttft,
                cap_hi=min(pre_cap, dec_cap), cap_lo=1 / (1 / pre_cap + 1 / dec_cap))

# The workload: 70B dense in FP8 with an FP8 KV cache; 1,500 prompt + 300 output tokens;
# TPOT target 50 ms, TTFT target 1 s, peak 20 requests/s.
WORKLOAD = dict(model="70B", prec="FP8", prompt=1500, output=300, kv_bytes=1.0, tpot_slo_ms=50)
PEAK_RPS, TTFT_SLO_S, UTIL = 20, 1.0, 0.7

def fleet(cap, tp):
    lo = math.ceil(PEAK_RPS / (cap["cap_hi"] * UTIL)) * tp
    hi = math.ceil(PEAK_RPS / (cap["cap_lo"] * UTIL)) * tp
    return lo, hi

print(f"{'replica':<14}{'KV pool':>8}{'batch':>6}{'step':>8}{'bound':>9}{'dec tok/s':>10}{'TTFT floor':>11}{'req/s':>12}{'GPUs':>9}")
for g, tp in [("H200", 1), ("H100-SXM", 2), ("H20", 1), ("H20", 2)]:
    c = replica_capacity(g, WORKLOAD["model"], WORKLOAD["prec"], WORKLOAD["prompt"], WORKLOAD["output"],
                         tp=tp, kv_bytes=WORKLOAD["kv_bytes"], tpot_slo_ms=WORKLOAD["tpot_slo_ms"])
    lo, hi = fleet(c, tp)
    ok = "" if c["ttft_floor_s"] <= TTFT_SLO_S else " (misses TTFT)"
    print(f"{g + ' TP=' + str(tp):<14}{c['pool_gb']:>6.0f}GB{c['batch']:>6}{c['step_ms']:>6.0f}ms{c['bound']:>9}"
          f"{c['decode_tok_s']:>10,.0f}{c['ttft_floor_s']:>10.2f}s{c['cap_lo']:>6.2f}-{c['cap_hi']:<5.2f}{lo:>4}-{hi}{ok}")

Read the table:

- **Prefill binds on every card**, so the fleet scales with FLOP/s. An H200 and an H100 pair deliver the same compute per GPU, hence 7–8 vs 8 GPUs.
- **The H20 is compute-bound even in decode** (its ridge is 37, and a 70B model at batch 46 is past it), and a single H20 misses the 1 s TTFT floor outright. TP=2 isn't for memory, it's to halve prefill time; per GPU it serves no more. Result: **42–50 H20s against 7–8 H200s.**

> **🔵 Interview signal —** say the chain with its assumptions: *"70B in FP8 is 71 GB, so one H200 with 54 GB of cache; decode is memory-bound at ~31 ms a step; prefill of 1,500 tokens is 0.21 s and binds capacity at about 4 req/s per card; 20 req/s at 70% is 7–8 cards. The input I trust least is MFU, so I'd load-test that first."*

In [ ]:
# Change the shape: a decode-heavy generator (200 prompt tokens, 1,000 output tokens).
for g, tp in [("H200", 1), ("H20", 1)]:
    c = replica_capacity(g, "70B", "FP8", 200, 1000, tp=tp, kv_bytes=1.0, tpot_slo_ms=50)
    lo, hi = fleet(c, tp)
    print(f"{g:<5} batch {c['batch']:>3}  {c['bound']:<7}  {c['decode_tok_s']:>6,.0f} tok/s  GPUs for {PEAK_RPS} req/s: {lo}-{hi}")

The gap narrows (5 vs 28–33) but the H20 is still compute-bound for a 70B dense model. It shines where FLOPs **and** bytes per token are small: small dense models, and MoE models with few active parameters.

## Step 5 — the bill

`cost per million output tokens = fleet $/hour ÷ (output tokens per hour ÷ 10⁶)`. The rental prices below are **assumptions for illustration** (as of 2026-11); plug in your own quote.

In [ ]:
ASSUMED_PRICE_PER_GPU_HOUR = {"H200": 3.50, "H100-SXM": 2.50, "H20": 1.20}   # illustrative only
out_tokens_per_hour = PEAK_RPS * WORKLOAD["output"] * 3600

for g, n in [("H200", 8), ("H100-SXM", 8), ("H20", 50)]:
    per_hour = n * ASSUMED_PRICE_PER_GPU_HOUR[g]
    print(f"{n:>2} x {g:<9} ${per_hour:>6.2f}/h   ${per_hour / (out_tokens_per_hour / 1e6):.2f} per million output tokens")

The cheapest card per hour is the most expensive per token for this shape. And this is at *peak*: off-peak the fleet idles unless it scales down, so the real cost per token is higher.

## An accelerator without a datasheet

Alibaba T-Head's PPU (Zhenwu 810E) is reported with 96 GB of HBM2e, PCIe 5.0, and 400 W, "comparable to an H20", but its bandwidth and FLOP/s aren't published (as of 2026-11). Fit is already done (it's the H20's row). For the rest, **bracket** the unknowns and plan on the range, then measure.

In [ ]:
GPUS["PPU-low"]  = (96, 1.6, 120, None)    # assumed lower bracket, NOT a published spec
GPUS["PPU-high"] = (96, 4.0, 300, None)    # assumed upper bracket, NOT a published spec

for g in ["PPU-low", "PPU-high"]:
    b1 = 1000 / decode_step_ms(g, "8B", "BF16", 1, 0)[0]
    pre = prefill_s(g, "8B", "BF16", 2000) * 1000
    print(f"{g:<9} 8B BF16: batch-1 {b1:>4.0f} tok/s, 2,000-token prefill {pre:>4.0f} ms")

A 2.5× range on decode and prefill is too wide to buy against. Two measurements on a loaner card (a memory copy for bandwidth and a large matrix multiply for FLOP/s, exactly what the appendix runs on a T4) collapse it. The same applies to Huawei's Ascend 910B/910C, whose public numbers are analyst estimates; also check the serving stack (vLLM's Ascend plugin, or Huawei's MindIE) supports your model.

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Compare **dense** FLOP/s | Compare a sparse figure with a dense one and pick the "2× faster" card |
| Fit weights + overhead + KV cache for the target batch | Fit weights and discover the batch is 2 |
| Read `num_key_value_heads` and `head_dim` from the config | Use attention heads and overestimate the cache 4–8× |
| Check the ridge point and compute prefill time | Size on bandwidth alone and miss TTFT on a compute-poor card |
| Fit MoE on **total** parameters | Assume the 22B active parameters are all that must fit |
| Bracket unpublished specs, then measure | Plan a fleet on a press-release number |
| Size for ~70% utilization | Plan for 100% and watch p99 explode on every burst |

## Optional appendix — measure a real GPU (needs a GPU)

> **⚠️ This appendix needs a GPU runtime and is skippable.** In Colab: **Runtime → Change runtime type → T4 GPU**. On CPU, read it and move on; the arithmetic above is the graded material.

Three measurements replace the three biggest assumptions: achievable **bandwidth** (MBU), achievable **FLOP/s** (MFU), and real batch-1 **decode speed** compared with the prediction.

In [ ]:
# Appendix cell 1 — GPU check. Stop here if this prints "no GPU".
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — the arithmetic above is what's graded; skip this appendix")

In [ ]:
# Appendix cell 2 — measure bandwidth (a big device-to-device copy reads and writes every byte once)
# and FP16 matmul FLOP/s. These are the two numbers to collect on any loaner card, PPU and Ascend included.
import torch, time

def timed(fn, iters):
    fn(); torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters

n_bytes = 1 << 30                                   # 1 GiB
src = torch.empty(n_bytes, dtype=torch.uint8, device="cuda")
dst = torch.empty_like(src)
copy_s = timed(lambda: dst.copy_(src), 20)
measured_bw = 2 * n_bytes / copy_s / 1e12            # read + write

n = 8192
a = torch.randn(n, n, dtype=torch.float16, device="cuda")
b = torch.randn(n, n, dtype=torch.float16, device="cuda")
mm_s = timed(lambda: a @ b, 10)
measured_tflops = 2 * n ** 3 / mm_s / 1e12

spec_bw, spec_tf = GPUS["T4"][1], GPUS["T4"][2]
print(f"bandwidth {measured_bw * 1000:.0f} GB/s  = {measured_bw / spec_bw:.0%} of the {spec_bw * 1000:.0f} GB/s spec (your MBU ceiling)")
print(f"FP16 matmul {measured_tflops:.1f} TFLOP/s = {measured_tflops / spec_tf:.0%} of the {spec_tf} TFLOP/s spec (your MFU ceiling)")
del src, dst, a, b; torch.cuda.empty_cache()

In [ ]:
# Appendix cell 3 — predict batch-1 decode for the model the server will load.
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"           # the one place the model name lives
MODELS["tiny"] = dict(params=1.1, layers=22, kv_heads=4, head_dim=64)   # from its config.json

weights = weights_gb("tiny", "BF16")                     # served in FP16 on a T4: same 2 bytes
predicted = measured_bw * 1e12 / (weights * 1e9)
print(f"{MODEL}: {weights:.1f} GB of weights -> predicted batch-1 ceiling ~{predicted:.0f} tok/s at measured bandwidth")

In [ ]:
# Appendix cell 4 — install vLLM + fix the torchaudio CUDA-version clash (same as notebook 02).
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"
# If the line above errors, Runtime -> Restart session, then rerun from cell 1 SKIPPING this %pip line.

In [ ]:
# Appendix cell 5 — launch the server. Its log prints the real KV pool ("Available KV cache memory")
# and "Maximum concurrency": compare them with kv_pool_gb() for a 16 GB card.
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--max-model-len", "2048", "--port", "8000", "--enforce-eager", "--gpu-memory-utilization", "0.85"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(90):
    if server.poll() is not None:
        print("server exited; log tail:"); print(open(LOG).read()[-3000:]); break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2); print("server is up"); break
    except Exception:
        time.sleep(5)

!grep -iE "KV cache memory|Maximum concurrency" /content/vllm.log | tail -3

In [ ]:
# Appendix cell 6 — measure batch-1 decode speed with streaming, and compare with the prediction.
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

def stream_once(max_tokens=200):
    t0 = time.perf_counter(); first = None; n = 0
    stream = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Write a long story about a lighthouse keeper."}],
        max_tokens=max_tokens, temperature=0.8, stream=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            n += 1
            first = first or time.perf_counter()
    end = time.perf_counter()
    return first - t0, n / (end - first)

stream_once(20)                                          # warm-up
ttft, tok_s = stream_once()
print(f"TTFT {ttft * 1000:.0f} ms, decode {tok_s:.0f} tok/s (chunks/s) vs predicted ceiling {predicted:.0f} tok/s "
      f"-> {tok_s / predicted:.0%} of the bandwidth ceiling")

Expect the measured number to sit well **below** the ceiling for a 1.1B model: with `--enforce-eager`, the per-step Python and kernel-launch overhead is comparable to the ~10 ms weight read. The formula is a ceiling that large models approach and tiny ones don't. That's itself a sizing lesson: for small models, server overhead, not bandwidth, can be the limit, so always measure.

In [ ]:
# Appendix cell 7 — free the GPU (a connected runtime still uses Colab GPU quota: disconnect when done).
server.terminate()
time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

## Exercises

1. **Your own fit table.** Add a model you care about to `MODELS` (read its `config.json`) and an accelerator from your cloud's price list to `GPUS`. Which is the smallest card that holds it with room for 16 concurrent 4,000-token conversations?
2. **FP8 KV cache.** Re-run the three-accelerator table with `kv_bytes=2.0` (a BF16 cache). Which replicas lose the most capacity, and why does the H200's change more than the H20's?
3. **Where the H20 wins.** Find a model and request shape (prompt and output tokens) where H20s need *fewer or equal* GPUs than H100s. Explain it with the ridge point.
4. **Close the bracket.** Suppose a PPU loaner measures 1.9 TB/s and 110 TFLOP/s FP16 with your appendix cells. Add it to `GPUS` with those values and size the 70B workload (BF16 weights don't fit 96 GB, so pick a precision). Does it meet the 1 s TTFT target with TP=1?
5. **Measured efficiencies.** Set `MBU` and `MFU` to what appendix cell 2 measured on the T4 and re-run steps 2–4. How much did the fleet sizes move?